## Model Comparison for Li-Air Overpotential

This notebook compares NGBoost, Gaussian Process Regression, and
TabPFN using the iteration 6 Li-air dataset. Inputs are
`CurrentDensity`, `O2`, and `CO2`.

### Workflow

1. Split by compound, reserving 25% of compound groups for testing.
2. Select NGBoost and GPR hyperparameters by five-fold grouped CV MAE.
3. Evaluate pooled out-of-fold predictions and training/test predictions
   using R², MAE, and RMSE.
4. Assess 95% prediction intervals using coverage (PICP95),
   mean width (MPIW95), and interval score (IS95).
5. Export CV/test predictions and comparison metrics ranked by CV MAE.

NGBoost and GPR intervals use predictive means and standard deviations;
TabPFN intervals use the 2.5th and 97.5th percentiles.
Hyperparameter selection and CV evaluation reuse the same folds;
this is not nested cross-validation.
TabPFN runs on CPU using the specified local checkpoint.
Run cells sequentially.

In [1]:
# ============================================================
# 1. Import packages
# ============================================================

import gc
import os

import numpy as np
import pandas as pd
import torch

from ngboost import NGBRegressor
from ngboost.distns import Normal
from ngboost.scores import LogScore

from tabpfn import TabPFNRegressor

from sklearn.base import clone
from sklearn.gaussian_process import (
    GaussianProcessRegressor
)
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF,
    WhiteKernel
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    GroupShuffleSplit
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

D:\app\miniconda\envs\mp311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 2. Basic settings
# ============================================================

DATA_FILE = "data/6_air.xlsx"
SHEET_NAME = "Sheet1"

TARGET_COL = "overpotential"

FEATURE_COLS = [
    "CurrentDensity",
    "O2",
    "CO2"
]

TEST_SIZE = 0.25
RANDOM_STATE = 42
INTERVAL_ALPHA = 0.05

TABPFN_MODEL_PATH = (
    "tabpfn_models/"
    "tabpfn-v2-regressor.ckpt"
)

OUTPUT_DIR = "model_comparison/air"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

In [3]:
# ============================================================
# 3. Load data
# ============================================================

data = pd.read_excel(
    DATA_FILE,
    sheet_name=SHEET_NAME,
    index_col=0
).reset_index(drop=True)

GROUP_COL = (
    "compound"
    if "compound" in data.columns
    else "compounds"
)

X = data[FEATURE_COLS]
y = data[TARGET_COL]
groups = data[GROUP_COL]

print("Data columns:", data.columns.tolist())
print("Group column:", GROUP_COL)
print("Target column:", TARGET_COL)
print("Total samples:", len(data))
print("Total compounds:", groups.nunique())
print("Feature columns:", FEATURE_COLS)

Data columns: ['compounds', 'CurrentDensity', 'O2', 'CO2', 'overpotential']
Group column: compounds
Target column: overpotential
Total samples: 499
Total compounds: 85
Feature columns: ['CurrentDensity', 'O2', 'CO2']


In [4]:
# ============================================================
# 4. Group-based train/test split
# ============================================================

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_index, test_index = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)

train_data = (
    data
    .iloc[train_index]
    .copy()
    .reset_index(drop=True)
)

test_data = (
    data
    .iloc[test_index]
    .copy()
    .reset_index(drop=True)
)

X_train = train_data[FEATURE_COLS]
X_test = test_data[FEATURE_COLS]

y_train = (
    train_data[TARGET_COL]
    .to_numpy()
)

y_test = (
    test_data[TARGET_COL]
    .to_numpy()
)

groups_train = (
    train_data[GROUP_COL]
    .to_numpy()
)

print("\nTraining samples:", len(train_data))
print("Test samples:", len(test_data))
print(
    "Training compounds:",
    train_data[GROUP_COL].nunique()
)
print(
    "Test compounds:",
    test_data[GROUP_COL].nunique()
)

overlap = (
    set(train_data[GROUP_COL])
    & set(test_data[GROUP_COL])
)

print("Overlapping compounds:", len(overlap))


Training samples: 377
Test samples: 122
Training compounds: 63
Test compounds: 22
Overlapping compounds: 0


In [5]:
# ============================================================
# 5. Group cross-validation
# ============================================================

group_cv = GroupKFold(
    n_splits=5
)

cv_splits = list(
    group_cv.split(
        X_train,
        y_train,
        groups=groups_train
    )
)


# ============================================================
# 6. Metric functions
# ============================================================

def point_metrics(
    y_true,
    y_pred
):

    r2 = r2_score(
        y_true,
        y_pred
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    return r2, mae, rmse


def uncertainty_metrics(
    y_true,
    lower,
    upper,
    alpha=0.05
):

    covered = (
        (y_true >= lower)
        & (y_true <= upper)
    )

    coverage = np.mean(
        covered
    )

    width = upper - lower

    mean_width = np.mean(
        width
    )

    lower_penalty = (
        (2.0 / alpha)
        * (lower - y_true)
        * (y_true < lower)
    )

    upper_penalty = (
        (2.0 / alpha)
        * (y_true - upper)
        * (y_true > upper)
    )

    interval_score = np.mean(
        width
        + lower_penalty
        + upper_penalty
    )

    return (
        coverage,
        mean_width,
        interval_score
    )


# ============================================================
# 7. Probabilistic prediction function
# ============================================================

def probabilistic_predict(
    model_name,
    model,
    X_input
):

    if model_name == "NGBoost":

        distribution = model.pred_dist(
            X_input
        )

        mean_prediction = np.asarray(
            distribution.loc
        ).reshape(-1)

        prediction_std = np.asarray(
            distribution.scale
        ).reshape(-1)

        lower = (
            mean_prediction
            - 1.959964 * prediction_std
        )

        upper = (
            mean_prediction
            + 1.959964 * prediction_std
        )

    elif model_name == "GPR":

        scaler = model.named_steps[
            "scaler"
        ]

        regressor = model.named_steps[
            "model"
        ]

        X_scaled = scaler.transform(
            X_input
        )

        mean_prediction, prediction_std = (
            regressor.predict(
                X_scaled,
                return_std=True
            )
        )

        mean_prediction = np.asarray(
            mean_prediction
        ).reshape(-1)

        prediction_std = np.asarray(
            prediction_std
        ).reshape(-1)

        lower = (
            mean_prediction
            - 1.959964 * prediction_std
        )

        upper = (
            mean_prediction
            + 1.959964 * prediction_std
        )

    elif model_name == "TabPFN":

        mean_prediction = model.predict(
            X_input,
            output_type="mean"
        )

        quantile_prediction = model.predict(
            X_input,
            output_type="quantiles",
            quantiles=[
                0.025,
                0.975
            ]
        )

        mean_prediction = np.asarray(
            mean_prediction
        ).reshape(-1)

        lower = np.asarray(
            quantile_prediction[0]
        ).reshape(-1)

        upper = np.asarray(
            quantile_prediction[1]
        ).reshape(-1)

        prediction_std = (
            upper - lower
        ) / (2.0 * 1.959964)

    else:

        raise ValueError(
            f"Unsupported model: {model_name}"
        )

    prediction_std = np.maximum(
        prediction_std,
        1e-8
    )

    return (
        mean_prediction,
        prediction_std,
        lower,
        upper
    )


In [6]:
# ============================================================
# 8. Cross-validation evaluation
# ============================================================

def cross_validation_predictions(
    model_name,
    model
):

    cv_true = []
    cv_mean = []
    cv_std = []
    cv_lower = []
    cv_upper = []

    for train_index, valid_index in cv_splits:

        fold_model = clone(model)

        X_fold_train = X_train.iloc[
            train_index
        ]

        y_fold_train = y_train[
            train_index
        ]

        X_fold_valid = X_train.iloc[
            valid_index
        ]

        y_fold_valid = y_train[
            valid_index
        ]

        fold_model.fit(
            X_fold_train,
            y_fold_train
        )

        (
            fold_mean,
            fold_std,
            fold_lower,
            fold_upper
        ) = probabilistic_predict(
            model_name,
            fold_model,
            X_fold_valid
        )

        cv_true.append(
            y_fold_valid
        )

        cv_mean.append(
            fold_mean
        )

        cv_std.append(
            fold_std
        )

        cv_lower.append(
            fold_lower
        )

        cv_upper.append(
            fold_upper
        )

        del fold_model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    cv_true = np.concatenate(
        cv_true
    )

    cv_mean = np.concatenate(
        cv_mean
    )

    cv_std = np.concatenate(
        cv_std
    )

    cv_lower = np.concatenate(
        cv_lower
    )

    cv_upper = np.concatenate(
        cv_upper
    )

    return (
        cv_true,
        cv_mean,
        cv_std,
        cv_lower,
        cv_upper
    )


# ============================================================
# 9. Complete model evaluation
# ============================================================

def evaluate_probabilistic_model(
    model_name,
    model
):

    (
        cv_true,
        cv_mean,
        cv_std,
        cv_lower,
        cv_upper
    ) = cross_validation_predictions(
        model_name,
        model
    )

    cv_r2, cv_mae, cv_rmse = (
        point_metrics(
            cv_true,
            cv_mean
        )
    )

    (
        cv_coverage,
        cv_width,
        cv_interval_score
    ) = uncertainty_metrics(
        cv_true,
        cv_lower,
        cv_upper,
        alpha=INTERVAL_ALPHA
    )

    model.fit(
        X_train,
        y_train
    )

    (
        train_mean,
        train_std,
        train_lower,
        train_upper
    ) = probabilistic_predict(
        model_name,
        model,
        X_train
    )

    (
        test_mean,
        test_std,
        test_lower,
        test_upper
    ) = probabilistic_predict(
        model_name,
        model,
        X_test
    )

    train_r2, train_mae, train_rmse = (
        point_metrics(
            y_train,
            train_mean
        )
    )

    (
        train_coverage,
        train_width,
        train_interval_score
    ) = uncertainty_metrics(
        y_train,
        train_lower,
        train_upper,
        alpha=INTERVAL_ALPHA
    )

    test_r2, test_mae, test_rmse = (
        point_metrics(
            y_test,
            test_mean
        )
    )

    (
        test_coverage,
        test_width,
        test_interval_score
    ) = uncertainty_metrics(
        y_test,
        test_lower,
        test_upper,
        alpha=INTERVAL_ALPHA
    )

    print(f"\n===== {model_name} =====")

    print(
        f"CV: R2={cv_r2:.6f} | "
        f"MAE={cv_mae:.6f} | "
        f"RMSE={cv_rmse:.6f} | "
        f"PICP95={cv_coverage:.6f} | "
        f"MPIW95={cv_width:.6f} | "
        f"IS95={cv_interval_score:.6f}"
    )

    print(
        f"Train: R2={train_r2:.6f} | "
        f"MAE={train_mae:.6f} | "
        f"RMSE={train_rmse:.6f} | "
        f"PICP95={train_coverage:.6f} | "
        f"MPIW95={train_width:.6f} | "
        f"IS95={train_interval_score:.6f}"
    )

    print(
        f"Test: R2={test_r2:.6f} | "
        f"MAE={test_mae:.6f} | "
        f"RMSE={test_rmse:.6f} | "
        f"PICP95={test_coverage:.6f} | "
        f"MPIW95={test_width:.6f} | "
        f"IS95={test_interval_score:.6f}"
    )

    result = {
        "Model": model_name,
        "CV_R2": cv_r2,
        "CV_MAE": cv_mae,
        "CV_RMSE": cv_rmse,
        "CV_PICP95": cv_coverage,
        "CV_MPIW95": cv_width,
        "CV_IS95": cv_interval_score,
        "Train_R2": train_r2,
        "Train_MAE": train_mae,
        "Train_RMSE": train_rmse,
        "Train_PICP95": train_coverage,
        "Train_MPIW95": train_width,
        "Train_IS95": train_interval_score,
        "Test_R2": test_r2,
        "Test_MAE": test_mae,
        "Test_RMSE": test_rmse,
        "Test_PICP95": test_coverage,
        "Test_MPIW95": test_width,
        "Test_IS95": test_interval_score
    }

    cv_prediction_result = pd.DataFrame({
        "y_true": cv_true,
        "y_pred": cv_mean,
        "y_std": cv_std,
        "lower_95": cv_lower,
        "upper_95": cv_upper
    })

    cv_prediction_result.to_csv(
        f"{OUTPUT_DIR}/{model_name}_cv_predictions.csv",
        index=False
    )

    test_prediction_result = pd.DataFrame({
        GROUP_COL: test_data[GROUP_COL],
        "y_true": y_test,
        "y_pred": test_mean,
        "y_std": test_std,
        "lower_95": test_lower,
        "upper_95": test_upper
    })

    test_prediction_result.to_csv(
        f"{OUTPUT_DIR}/{model_name}_test_predictions.csv",
        index=False
    )

    return model, result

In [7]:
# ============================================================
# 10. NGBoost hyperparameter selection
# ============================================================

ngb_base = DecisionTreeRegressor(
    max_depth=2,
    min_samples_leaf=3,
    random_state=RANDOM_STATE
)

ngboost_model = NGBRegressor(
    Dist=Normal,
    Score=LogScore,
    Base=ngb_base,
    random_state=RANDOM_STATE,
    verbose=False
)

ngboost_grid = {
    "n_estimators": [
        100,
        200,
        300
    ],
    "learning_rate": [
        0.01,
        0.03,
        0.05
    ],
    "minibatch_frac": [
        0.8,
        1.0
    ]
}

ngboost_search = GridSearchCV(
    estimator=ngboost_model,
    param_grid=ngboost_grid,
    scoring="neg_mean_absolute_error",
    cv=cv_splits,
    n_jobs=-1,
    refit=True
)

ngboost_search.fit(
    X_train,
    y_train
)

best_ngboost = (
    ngboost_search.best_estimator_
)

print(
    "NGBoost best parameters:",
    ngboost_search.best_params_
)

NGBoost best parameters: {'learning_rate': 0.01, 'minibatch_frac': 0.8, 'n_estimators': 300}


In [8]:
# ============================================================
# 11. GPR hyperparameter selection
# ============================================================

gpr_kernel = (
    ConstantKernel(
        constant_value=1.0,
        constant_value_bounds=(1e-2, 1e2)
    )
    * RBF(
        length_scale=1.0,
        length_scale_bounds=(1e-2, 1e2)
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-6, 1e0)
    )
)

gpr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        GaussianProcessRegressor(
            kernel=gpr_kernel,
            normalize_y=True,
            n_restarts_optimizer=5,
            random_state=RANDOM_STATE
        )
    )
])

gpr_grid = {
    "model__alpha": [
        1e-8,
        1e-6,
        1e-3,
        1e-2
    ]
}

gpr_search = GridSearchCV(
    estimator=gpr_model,
    param_grid=gpr_grid,
    scoring="neg_mean_absolute_error",
    cv=cv_splits,
    n_jobs=-1,
    refit=True
)

gpr_search.fit(
    X_train,
    y_train
)

best_gpr = (
    gpr_search.best_estimator_
)

print(
    "GPR best parameters:",
    gpr_search.best_params_
)


GPR best parameters: {'model__alpha': 1e-08}


In [9]:
# ============================================================
# 12. TabPFN
# ============================================================

best_tabpfn = TabPFNRegressor(
    device="cpu",
    model_path=TABPFN_MODEL_PATH,
    random_state=RANDOM_STATE
)

In [10]:
# ============================================================
# 13. Evaluate all probabilistic models
# ============================================================

best_ngboost, ngboost_result = (
    evaluate_probabilistic_model(
        model_name="NGBoost",
        model=best_ngboost
    )
)

best_gpr, gpr_result = (
    evaluate_probabilistic_model(
        model_name="GPR",
        model=best_gpr
    )
)

best_tabpfn, tabpfn_result = (
    evaluate_probabilistic_model(
        model_name="TabPFN",
        model=best_tabpfn
    )
)


===== NGBoost =====
CV: R2=0.831686 | MAE=0.117542 | RMSE=0.173200 | PICP95=0.896552 | MPIW95=0.471228 | IS95=0.997038
Train: R2=0.909973 | MAE=0.089591 | RMSE=0.126670 | PICP95=0.976127 | MPIW95=0.479271 | IS95=0.556794
Test: R2=0.874750 | MAE=0.098048 | RMSE=0.151917 | PICP95=0.877049 | MPIW95=0.497370 | IS95=1.132411

===== GPR =====
CV: R2=0.815858 | MAE=0.121435 | RMSE=0.181160 | PICP95=0.915119 | MPIW95=0.565800 | IS95=1.179850
Train: R2=0.896216 | MAE=0.096923 | RMSE=0.136004 | PICP95=0.957560 | MPIW95=0.567777 | IS95=0.763096
Test: R2=0.874557 | MAE=0.099179 | RMSE=0.152034 | PICP95=0.909836 | MPIW95=0.568177 | IS95=1.012560


D:\app\miniconda\envs\mp311\Lib\site-packages\tabpfn\validation.py:150: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(
D:\app\miniconda\envs\mp311\Lib\site-packages\tabpfn\validation.py:150: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(
D:\app\miniconda\envs\mp311\Lib\site-packages\tabpfn\validation.py:150: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(
D:\app\miniconda\envs\mp311\Lib\site-packages\tabpfn\validation.py:150: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/Prior


===== TabPFN =====
CV: R2=0.820531 | MAE=0.118192 | RMSE=0.178847 | PICP95=0.893899 | MPIW95=0.576769 | IS95=1.071564
Train: R2=0.905374 | MAE=0.088124 | RMSE=0.129865 | PICP95=0.978780 | MPIW95=0.476359 | IS95=0.580514
Test: R2=0.875396 | MAE=0.098400 | RMSE=0.151525 | PICP95=0.901639 | MPIW95=0.501841 | IS95=1.098522


In [11]:
# ============================================================
# 14. Model comparison
# ============================================================

comparison_results = pd.DataFrame([
    ngboost_result,
    gpr_result,
    tabpfn_result
])

comparison_results = (
    comparison_results
    .sort_values(
        "CV_MAE",
        ascending=True
    )
    .reset_index(drop=True)
)

print("\nFinal model comparison:")

print(
    comparison_results[
        [
            "Model",
            "CV_R2",
            "CV_MAE",
            "CV_RMSE",
            "CV_PICP95",
            "CV_MPIW95",
            "CV_IS95",
            "Test_R2",
            "Test_MAE",
            "Test_RMSE",
            "Test_PICP95",
            "Test_MPIW95",
            "Test_IS95"
        ]
    ]
)

comparison_results.to_csv(
    f"{OUTPUT_DIR}/probabilistic_model_comparison.csv",
    index=False
)



Final model comparison:
     Model     CV_R2    CV_MAE   CV_RMSE  CV_PICP95  CV_MPIW95   CV_IS95  \
0  NGBoost  0.831686  0.117542  0.173200   0.896552   0.471228  0.997038   
1   TabPFN  0.820531  0.118192  0.178847   0.893899   0.576769  1.071564   
2      GPR  0.815858  0.121435  0.181160   0.915119   0.565800  1.179850   

    Test_R2  Test_MAE  Test_RMSE  Test_PICP95  Test_MPIW95  Test_IS95  
0  0.874750  0.098048   0.151917     0.877049     0.497370   1.132411  
1  0.875396  0.098400   0.151525     0.901639     0.501841   1.098522  
2  0.874557  0.099179   0.152034     0.909836     0.568177   1.012560  
